# Global Country Info Analysis - AI/ML/DL

This notebook provides comprehensive analysis of global country data using:
- **Artificial Intelligence (AI)**: Intelligent pattern recognition and insights
- **Machine Learning (ML)**: Regression, Classification, and Clustering
- **Deep Learning (DL)**: Neural Networks for complex predictions

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error, r2_score, classification_report, silhouette_score
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")

## 1. Load and Explore Dataset

In [ ]:
# Load the dataset
df = pd.read_csv('world-data-2023.csv')

print(f"Dataset Shape: {df.shape}")
print(f"Number of Countries: {df.shape[0]}")
print(f"Number of Features: {df.shape[1]}")

# Display first few rows
df.head()

In [ ]:
# Dataset info
df.info()

In [ ]:
# Check for missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing': missing,
    'Percentage': missing_pct
}).sort_values('Missing', ascending=False)

missing_df[missing_df['Missing'] > 0]

## 2. Data Preprocessing

In [ ]:
# Preprocess the data
df_processed = df.copy()

# Convert percentage strings to floats
percentage_cols = [col for col in df_processed.columns if '%' in str(col)]
for col in percentage_cols:
    if col in df_processed.columns:
        df_processed[col] = df_processed[col].astype(str).str.replace('%', '').str.replace(',', '')
        df_processed[col] = pd.to_numeric(df_processed[col], errors='coerce')

# Clean currency columns
currency_cols = ['Gasoline Price', 'GDP', 'Minimum wage']
for col in currency_cols:
    if col in df_processed.columns:
        df_processed[col] = df_processed[col].astype(str).str.replace('$', '').str.replace(',', '').str.strip()
        df_processed[col] = pd.to_numeric(df_processed[col], errors='coerce')

# Clean numeric columns with commas
for col in df_processed.columns:
    if df_processed[col].dtype == 'object':
        try:
            df_processed[col] = df_processed[col].astype(str).str.replace(',', '')
            df_processed[col] = pd.to_numeric(df_processed[col], errors='ignore')
        except:
            pass

# Select only numeric columns
numeric_cols = df_processed.select_dtypes(include=[np.number]).columns.tolist()
df_numeric = df_processed[numeric_cols].copy()

# Fill missing values with median
df_numeric = df_numeric.fillna(df_numeric.median())

print(f"Processed data shape: {df_numeric.shape}")
print(f"Numeric features: {len(numeric_cols)}")

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Statistical summary
df_numeric.describe()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(16, 12))
correlation = df_numeric.corr()
sns.heatmap(correlation, cmap='coolwarm', center=0, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Heatmap', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution plots for key features
key_features = ['Life expectancy', 'GDP', 'Population', 'Birth Rate', 'Infant mortality']
key_features = [f for f in key_features if f in df_numeric.columns]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(key_features):
    if i < len(axes):
        axes[i].hist(df_numeric[col], bins=30, edgecolor='black', alpha=0.7)
        axes[i].set_title(f'Distribution of {col}', fontweight='bold')
        axes[i].set_xlabel(col)
        axes[i].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## 4. Machine Learning - Regression (Life Expectancy Prediction)

In [ ]:
# Prepare data for regression
if 'Life expectancy' in df_numeric.columns:
    target = 'Life expectancy'
    df_reg = df_numeric.dropna(subset=[target])
    
    features = [col for col in df_reg.columns if col != target]
    X = df_reg[features]
    y = df_reg[target]
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Train Linear Regression
    lr_model = LinearRegression()
    lr_model.fit(X_train_scaled, y_train)
    lr_pred = lr_model.predict(X_test_scaled)
    
    # Train Random Forest
    rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_model.fit(X_train_scaled, y_train)
    rf_pred = rf_model.predict(X_test_scaled)
    
    # Results
    print("Linear Regression:")
    print(f"  MSE: {mean_squared_error(y_test, lr_pred):.2f}")
    print(f"  R² Score: {r2_score(y_test, lr_pred):.4f}")
    
    print("\nRandom Forest Regression:")
    print(f"  MSE: {mean_squared_error(y_test, rf_pred):.2f}")
    print(f"  R² Score: {r2_score(y_test, rf_pred):.4f}")
    
    # Plot predictions
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    axes[0].scatter(y_test, lr_pred, alpha=0.6)
    axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
    axes[0].set_xlabel('Actual Life Expectancy')
    axes[0].set_ylabel('Predicted Life Expectancy')
    axes[0].set_title('Linear Regression Predictions')
    
    axes[1].scatter(y_test, rf_pred, alpha=0.6, color='green')
    axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
    axes[1].set_xlabel('Actual Life Expectancy')
    axes[1].set_ylabel('Predicted Life Expectancy')
    axes[1].set_title('Random Forest Predictions')
    
    plt.tight_layout()
    plt.show()
    
    # Feature importance
    feature_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': rf_model.feature_importances_
    }).sort_values('importance', ascending=False).head(10)
    
    plt.figure(figsize=(10, 6))
    plt.barh(feature_importance['feature'], feature_importance['importance'])
    plt.xlabel('Importance')
    plt.title('Top 10 Most Important Features for Life Expectancy')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

## 5. Machine Learning - Clustering (Country Grouping)

In [ ]:
# K-Means Clustering
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_numeric)

# Elbow method
inertias = []
K_range = range(2, 11)
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)

# Plot elbow curve
plt.figure(figsize=(10, 6))
plt.plot(K_range, inertias, 'bo-')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal k')
plt.grid(True)
plt.show()

# Perform clustering with optimal k
optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

print(f"Silhouette Score: {silhouette_score(X_scaled, clusters):.4f}")

# Cluster distribution
unique, counts = np.unique(clusters, return_counts=True)
for cluster, count in zip(unique, counts):
    print(f"Cluster {cluster}: {count} countries")

# PCA for visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(12, 8))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=clusters, cmap='viridis', s=100, alpha=0.6)
plt.colorbar(scatter, label='Cluster')
plt.xlabel(f'First Principal Component ({pca.explained_variance_ratio_[0]:.2%} variance)')
plt.ylabel(f'Second Principal Component ({pca.explained_variance_ratio_[1]:.2%} variance)')
plt.title('Country Clusters Visualization (PCA)')
plt.grid(True, alpha=0.3)
plt.show()

## 6. Machine Learning - Classification (Development Level)

In [ ]:
# Create development categories
if 'Life expectancy' in df_numeric.columns and 'GDP' in df_numeric.columns:
    df_clf = df_numeric.copy()
    df_clf = df_clf.dropna(subset=['Life expectancy', 'GDP'])
    
    life_high = df_clf['Life expectancy'].quantile(0.75)
    life_low = df_clf['Life expectancy'].quantile(0.25)
    gdp_high = df_clf['GDP'].quantile(0.75)
    gdp_low = df_clf['GDP'].quantile(0.25)
    
    def classify_development(row):
        if row['Life expectancy'] >= life_high and row['GDP'] >= gdp_high:
            return 2  # High Development
        elif row['Life expectancy'] >= life_low and row['GDP'] >= gdp_low:
            return 1  # Medium Development
        else:
            return 0  # Low Development
    
    df_clf['Development_Level'] = df_clf.apply(classify_development, axis=1)
    
    # Prepare features and target
    target = 'Development_Level'
    features = [col for col in df_clf.columns if col != target]
    X = df_clf[features]
    y = df_clf[target]
    
    # Split and scale
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Train classifier
    rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_clf.fit(X_train_scaled, y_train)
    rf_pred = rf_clf.predict(X_test_scaled)
    
    print("Classification Report:")
    print(classification_report(y_test, rf_pred, target_names=['Low', 'Medium', 'High']))
    
    print(f"\nAccuracy: {rf_clf.score(X_test_scaled, y_test):.4f}")
    
    # Distribution
    plt.figure(figsize=(10, 6))
    development_counts = y.value_counts().sort_index()
    plt.bar(['Low', 'Medium', 'High'], development_counts.values)
    plt.xlabel('Development Level')
    plt.ylabel('Number of Countries')
    plt.title('Distribution of Countries by Development Level')
    plt.show()

## 7. Deep Learning - Neural Network (GDP Prediction)

In [ ]:
# Deep Learning with TensorFlow/Keras
try:
    from tensorflow import keras
    from keras.models import Sequential
    from keras.layers import Dense, Dropout
    
    if 'GDP' in df_numeric.columns:
        target = 'GDP'
        df_dl = df_numeric.dropna(subset=[target])
        
        features = [col for col in df_dl.columns if col != target]
        X = df_dl[features]
        y = df_dl[target]
        
        # Split data
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        
        # Scale
        scaler_X = StandardScaler()
        scaler_y = StandardScaler()
        X_train_scaled = scaler_X.fit_transform(X_train)
        X_test_scaled = scaler_X.transform(X_test)
        y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).flatten()
        y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1)).flatten()
        
        # Build Neural Network
        model = Sequential([
            Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)),
            Dropout(0.3),
            Dense(64, activation='relu'),
            Dropout(0.3),
            Dense(32, activation='relu'),
            Dropout(0.2),
            Dense(16, activation='relu'),
            Dense(1)
        ])
        
        model.compile(optimizer='adam', loss='mse', metrics=['mae'])
        
        print("Model Architecture:")
        model.summary()
        
        # Train
        history = model.fit(
            X_train_scaled, y_train_scaled,
            epochs=100,
            batch_size=16,
            validation_split=0.2,
            verbose=1
        )
        
        # Evaluate
        y_pred_scaled = model.predict(X_test_scaled)
        y_pred = scaler_y.inverse_transform(y_pred_scaled)
        
        mse = mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        
        print(f"\nTest MSE: {mse:.2e}")
        print(f"Test R² Score: {r2:.4f}")
        
        # Plot training history
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        
        axes[0].plot(history.history['loss'], label='Training Loss')
        axes[0].plot(history.history['val_loss'], label='Validation Loss')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Loss')
        axes[0].set_title('Model Loss')
        axes[0].legend()
        axes[0].grid(True)
        
        axes[1].scatter(y_test, y_pred, alpha=0.6)
        axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
        axes[1].set_xlabel('Actual GDP')
        axes[1].set_ylabel('Predicted GDP')
        axes[1].set_title('Neural Network Predictions')
        
        plt.tight_layout()
        plt.show()
        
except ImportError:
    print("TensorFlow/Keras not available. Please install: pip install tensorflow")

## 8. Summary and Insights

This notebook demonstrated:

### Machine Learning:
1. **Regression**: Predicted life expectancy using various country features
2. **Clustering**: Grouped countries into clusters based on their characteristics
3. **Classification**: Classified countries by development level

### Deep Learning:
- Built a neural network to predict GDP with multiple layers and dropout regularization

### Key Insights:
- Countries can be effectively grouped based on their socio-economic indicators
- Life expectancy is strongly correlated with GDP, healthcare, and education metrics
- Neural networks can capture complex non-linear relationships in the data